# PEMS-SF 1D-CNN 训练笔记 (V2.2.4 - 7类全周 + 梯度特征 + 错误分类数据)
目标： 利用 1D-CNN 的局部特征提取能力，结合原始流量与梯度信息，解决周二/周三混淆问题。

数据： PEMS-SF，双通道输入 (Raw Flow + Gradient)。

集成了之前讨论的所有增强手段：  

特征融合 (Feature Fusion): 输入双通道序列 (Flow + Gradient) + 统计特征 (Mean, Std, Max)。  

SE-Block: 增强 CNN 对关键特征的注意力。  

Focal Loss: 解决样本不平衡，专治“周三隐形”问题。  

数据增强 (Data Augmentation): 训练时加入随机噪声，提高泛化能力。  

强正则化: Dropout 提高到 0.5，防止过拟合。  

正确的数据集划分: 确保训练集有噪声，验证集/测试集干净。

# 与v2.2.0相比核心变更点：

Dropout 回调： 从 0.5 降回 0.25（接近 V2.1.3 的状态，保住周二特征）。

微量噪声： 降至 0.005 (0.5%)，既防止过拟合又不破坏均值特征。

智能加权 (Smart Weights): 同时保护 Tue/Wed/Thu，而不是只盯着 Wed。

进一步提高周三权重

In [8]:
# ==========================================
# Cell 1: 基础依赖与配置
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')

# 输出目录 V2.2.1
CM_DIR = ROOT / 'confusion_matrices224'
CM_DIR.mkdir(exist_ok=True)
MODEL_DIR = ROOT / 'models224' 
MODEL_DIR.mkdir(exist_ok=True)

# 随机种子 - 保证可复现性
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Dir: {CM_DIR}')

Device: cuda
Output Dir: confusion_matrices224


In [9]:
# 训练与解释的可复现性与确定性设置
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [10]:
# ==========================================
# Cell 2: 数据读取、预处理与增强 (V2.2.1 - 微量噪声)
# ==========================================

# 路径配置
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
stations_list_path = ROOT / 'data' / 'stations_list'
groups_index_path = ROOT / 'data' / 'processed' / 'pems_sf_groups_index.json'

# 基础解析函数
def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

# 加载元数据
labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]

# 加载分组掩码
group_station_masks = {}
if groups_index_path.exists():
    processed = json.loads(groups_index_path.read_text(encoding='utf-8'))
    groups_index = processed.get('groups_index', {})
    sid_to_pos = {sid: i for i, sid in enumerate(station_ids)}
    for g in ['g1','g2','g3','g4','g5']:
        mask = np.zeros(len(station_ids), dtype=bool)
        for sid in groups_index.get(g, []):
            if sid in sid_to_pos: mask[sid_to_pos[sid]] = True
        group_station_masks[g] = mask
    print('分组掩码加载完毕。')
else:
    print('警告：未找到分组文件，将使用全量数据。')
    group_station_masks = {'all': np.ones(len(station_ids), dtype=bool)}

# === 核心修改：曲线清洗与特征工程 ===
def _process_curve(curve: np.ndarray):
    # 1. 长度对齐与插值
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any():
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)

    # 2. 归一化 (Min-Max 到 0-1)
    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    # === 计算统计特征 (Fusion) ===
    mean_val = np.mean(norm_curve)
    std_val  = np.std(norm_curve)
    log_max_val = np.log1p(max_val) / 10.0 
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    # 3. 计算梯度
    grad_curve = np.gradient(norm_curve)
    
    # 4. 堆叠通道
    result = np.stack([norm_curve, grad_curve], axis=0)
    return result.astype(np.float32), stats

# Dataset 定义
# --- Cell 2: 在 PEMS_CNNDataset 内做最小改动：samples 里带 meta，并在 __getitem__ 返回 ---

class PEMS_CNNDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray, station_ids: list, augment=False, split_name=""):
        self.samples = []
        self.augment = augment
        self.split_name = split_name

        station_ids_arr = np.asarray(station_ids, dtype=int)

        # 依据 mask 先算出“过滤后”的 station_id 列表（保证与 mat[use_mask] 行对齐）
        if use_mask is not None and use_mask.shape[0] == station_ids_arr.shape[0]:
            kept_station_ids = station_ids_arr[use_mask]
            kept_station_pos = np.nonzero(use_mask)[0]  # station 在 stations_list 的位置
        else:
            kept_station_ids = station_ids_arr
            kept_station_pos = np.arange(station_ids_arr.shape[0], dtype=int)

        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue

            y = int(labels[day_i]) - 1
            if y < 0 or y > 6: continue

            if use_mask is not None and use_mask.shape[0] == mat.shape[0]:
                mat = mat[use_mask, :]

            # 对齐校验
            if mat.shape[0] != kept_station_ids.shape[0]:
                continue

            for local_sidx in range(mat.shape[0]):
                two_channel_seq, stats = _process_curve(mat[local_sidx, :144])
                if not np.isfinite(two_channel_seq).all(): 
                    continue

                meta = {
                    "station_id": int(kept_station_ids[local_sidx]),
                    "station_pos": int(kept_station_pos[local_sidx]),
                    "day_i": int(day_i),
                    "split": self.split_name,
                }
                self.samples.append((two_channel_seq, stats, y, meta))

        self.n = len(self.samples)

    def __len__(self): 
        return self.n

    def __getitem__(self, idx):
        seq, stats, y, meta = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)

        if self.augment:
            scale = random.uniform(0.95, 1.05)
            seq_tensor = seq_tensor * scale
            shift_steps = random.randint(-2, 2)
            if shift_steps != 0:
                seq_tensor = torch.roll(seq_tensor, shifts=shift_steps, dims=1)
            noise = torch.randn_like(seq_tensor) * 0.005
            seq_tensor = seq_tensor + noise

        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long), meta

分组掩码加载完毕。


In [11]:
# ==========================================
# Cell 3: 构建DataLoader (V2.2.1)
# ==========================================
group_loaders = {}
for g, mask in group_station_masks.items():
    print(f'准备分组 {g} 数据...')
    
    # 1. 实例化两个数据集：一个带增强(Train)，一个不带(Val)
    ds_train_source = PEMS_CNNDataset(
        train_path, labels_train, mask,
        station_ids=station_ids, augment=True, split_name="train"
    )
    ds_val_source = PEMS_CNNDataset(
        train_path, labels_train, mask,
        station_ids=station_ids, augment=False, split_name="val"
    )
    ds_test = PEMS_CNNDataset(
        test_path, labels_test, mask,
        station_ids=station_ids, augment=False, split_name="test"
    )
    
    # 2. 生成随机划分的索引
    total_len = len(ds_train_source)
    n_train = int(total_len * 0.8)
    indices = torch.randperm(total_len, generator=torch.Generator().manual_seed(42)).tolist()
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]
    
    # 3. 创建 Subset
    train_subset = Subset(ds_train_source, train_indices)
    val_subset   = Subset(ds_val_source, val_indices)
    
    group_loaders[g] = {
        'train': DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0),
        'val':   DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=0),
        'test':  DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=0),
    }
    print(f'  - Train: {len(train_subset)} (Augmented), Val: {len(val_subset)} (Clean), Test: {len(ds_test)}')

准备分组 g1 数据...
  - Train: 24000 (Augmented), Val: 6000 (Clean), Test: 25920
准备分组 g2 数据...
  - Train: 25600 (Augmented), Val: 6400 (Clean), Test: 27648
准备分组 g3 数据...
  - Train: 14400 (Augmented), Val: 3600 (Clean), Test: 15552
准备分组 g4 数据...
  - Train: 23400 (Augmented), Val: 5850 (Clean), Test: 25272
准备分组 g5 数据...
  - Train: 8900 (Augmented), Val: 2225 (Clean), Test: 9612


## 模型定义：模型定义 (SE + Fusion + High Dropout)

In [12]:
# ==========================================
# Cell 4: 1D-CNN + SE + Fusion (V2.2.1 - 适度 Dropout)
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

# 修改 Simple1DCNN 结构
class Simple1DCNN(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super(Simple1DCNN, self).__init__()
        
        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, 32, 7, padding=3),
            nn.BatchNorm1d(32), 
            nn.ReLU(), 
            nn.MaxPool1d(2)
        )
        self.se1 = SEBlock(32, reduction=4) 
        # 加入 Spatial Dropout，丢弃率较低 (0.1)，防止过度丢失信息
        self.drop1 = nn.Dropout1d(p=0.1) 

        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64), 
            nn.ReLU(), 
            nn.MaxPool1d(2)
        )
        self.se2 = SEBlock(64, reduction=8)
        self.drop2 = nn.Dropout1d(p=0.1)

        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128), 
            nn.ReLU()
        )
        self.se3 = SEBlock(128, reduction=16)

        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Classification Head
        self.fc = nn.Sequential(
            # 依然保留普通 Dropout 保护全连接层
            nn.Dropout(0.25), 
            nn.Linear(131, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        x = self.block1(x); x = self.se1(x); x = self.drop1(x) # 应用 drop1
        x = self.block2(x); x = self.se2(x); x = self.drop2(x) # 应用 drop2
        x = self.block3(x); x = self.se3(x)
        
        x = self.gap(x).view(x.size(0), -1) 
        combined = torch.cat([x, stats], dim=1) 
        logits = self.fc(combined)
        return logits

## 训练循环 (Focal Loss + Early Stopping)

In [13]:
# ==========================================
# Cell 5: 训练循环 (V2.2.1 - 保护性加权)
# ==========================================

# 修改 train_model 函数
def train_model(loaders, group_name, epochs=60): # 建议 epochs 提高到 60或80
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    # 稍微提高 weight_decay (L2正则化)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) 
    
    # 加入余弦退火学习率调度器 (让学习率慢慢降到接近 0)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    # 类别权重保持 V2.2.1 的最佳设定
    # Mon(0), Tue(1), Wed(2), Thu(3), Fri(4), Sat(5), Sun(6)
    # Mon/Sat/Sun: 1.0 (本来就准，不用管)
    # Tue/Thu: 2.0 (给它们加盾，防止被周五吃掉，也防止被周三权重压垮)
    # Wed: 4.0 (重点扶持，一定要把它从 Tue/Thu 堆里找出来)  从2.2.2版本的3进一步提升
    # Fri: 1.5 (稍微惩罚一下它的霸权，但别太狠，防止它把周四吞了)
    weights = torch.tensor([1.0, 2.0, 4.0, 2.0, 1.5, 1.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    best_acc = 0.0
    best_state = None
    patience = 15 # 容忍度适当调大，因为 scheduler 会在后期发力
    counter = 0
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y, _meta in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, stats)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # 每一个 epoch 结束更新学习率
        scheduler.step() 
        
        # Validation 逻辑保持不变...
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, stats, y, _meta in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                out = model(x, stats)
                preds = torch.argmax(out, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        # ... (和之前一样计算 val_acc)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            
        if epoch % 5 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"[{group_name}] Epoch {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%} | LR: {current_lr:.6f}")
        
        if counter >= patience:
            print(f"[{group_name}] Early stopping at epoch {epoch}")
            break
            
    return best_state, best_acc

# 开始训练
group_models = {}
for g, loaders in group_loaders.items():
    print(f"\n>>> 开始训练分组: {g}")
    state, acc = train_model(loaders, g, epochs=60) 
    group_models[g] = state
    print(f"[{g}] 训练完成，最佳验证准确率: {acc:.2%}")
    if state:
        torch.save(state, MODEL_DIR / f'cnn_v224_{g}.pth')

print("\n所有模型训练结束。")


>>> 开始训练分组: g1
[g1] Epoch 5/60 | Loss: 1.4842 | Val Acc: 39.38% | LR: 0.000983
[g1] Epoch 10/60 | Loss: 1.4227 | Val Acc: 47.75% | LR: 0.000933
[g1] Epoch 15/60 | Loss: 1.3951 | Val Acc: 48.43% | LR: 0.000854
[g1] Epoch 20/60 | Loss: 1.3763 | Val Acc: 55.25% | LR: 0.000750
[g1] Epoch 25/60 | Loss: 1.3575 | Val Acc: 55.50% | LR: 0.000630
[g1] Epoch 30/60 | Loss: 1.3456 | Val Acc: 57.35% | LR: 0.000501
[g1] Epoch 35/60 | Loss: 1.3304 | Val Acc: 56.40% | LR: 0.000371
[g1] Epoch 40/60 | Loss: 1.3225 | Val Acc: 55.83% | LR: 0.000251
[g1] Epoch 45/60 | Loss: 1.3155 | Val Acc: 58.05% | LR: 0.000147
[g1] Epoch 50/60 | Loss: 1.3089 | Val Acc: 58.45% | LR: 0.000068
[g1] Early stopping at epoch 52
[g1] 训练完成，最佳验证准确率: 58.73%

>>> 开始训练分组: g2
[g2] Epoch 5/60 | Loss: 1.4980 | Val Acc: 38.56% | LR: 0.000983
[g2] Epoch 10/60 | Loss: 1.4473 | Val Acc: 44.44% | LR: 0.000933
[g2] Epoch 15/60 | Loss: 1.4243 | Val Acc: 46.38% | LR: 0.000854
[g2] Epoch 20/60 | Loss: 1.4019 | Val Acc: 52.73% | LR: 0.000750
[g

## 评估与阈值
可视化 - 组合混淆矩阵 (V2.2.1)

In [15]:
# ==========================================
# Cell 6: 可视化 - 组合混淆矩阵 (V2.2.1)
# ==========================================

# --- Cell 6: get_predictions 增强：返回 metas ---

def get_predictions(model_state, loader):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    y_true, y_pred, metas = [], [], []

    with torch.no_grad():
        for x, stats, y, meta in loader:
            x = x.to(DEVICE)
            stats = stats.to(DEVICE)
            logits = model(x, stats)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            # meta 通常是 list[dict] 或 dict[list]
            if isinstance(meta, list):
                metas.extend(meta)
            elif isinstance(meta, dict):
                bs = len(y)
                for i in range(bs):
                    metas.append({k: meta[k][i] for k in meta})
            else:
                metas.extend(list(meta))

    return np.array(y_true), np.array(y_pred), metas

def plot_combined_matrix(mode='test'):
    print(f"正在生成 {mode} 集混淆矩阵组合图 (V2.2.1)...")
    
    layout = {
        'g1': (0, 0), 'g2': (0, 1), 'g3': (0, 2),
        'g4': (1, 0), 'g5': (1, 1), 'global': (1, 2)
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    global_true = []
    global_pred = []
    
    for g in ['g1', 'g2', 'g3', 'g4', 'g5']:
        if g not in group_models: continue
        
        loader = group_loaders[g][mode]
        y_t, y_p, _ = get_predictions(group_models[g], loader)
        global_true.extend(y_t)
        global_pred.extend(y_p)
        
        cm = confusion_matrix(y_t, y_p)
        cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
        
        row, col = layout[g]
        ax = axes[row, col]
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
        ax.set_title(f"{g.upper()} ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')

    # Global
    if len(global_true) > 0:
        cm_glob = confusion_matrix(global_true, global_pred)
        cm_glob_norm = cm_glob.astype('float') / (cm_glob.sum(axis=1)[:, np.newaxis] + 1e-8)
        
        row, col = layout['global']
        ax = axes[row, col]
        sns.heatmap(cm_glob_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=True, ax=ax)
        ax.set_title(f"Global All Groups ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')
    
    plt.suptitle(f'{mode.capitalize()} Set Confusion Matrices (Final Fusion V2.2.3)', 
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = CM_DIR / f'combined_{mode}_matrices_v224.png'
    plt.savefig(save_path, dpi=300)
    print(f"图表已保存: {save_path}")
    plt.close()

# 执行绘图
plot_combined_matrix('val')
plot_combined_matrix('test')

正在生成 val 集混淆矩阵组合图 (V2.2.1)...


图表已保存: confusion_matrices224/combined_val_matrices_v224.png
正在生成 test 集混淆矩阵组合图 (V2.2.1)...
图表已保存: confusion_matrices224/combined_test_matrices_v224.png


In [16]:
# ==========================================
# Cell 7: 导出逐样本预测（station_id + day_i）
# ==========================================
LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def export_predictions_station_day(mode="test"):
    rows = []
    for g in ["g1","g2","g3","g4","g5"]:
        if g not in group_models: 
            continue
        y_t, y_p, metas = get_predictions(group_models[g], group_loaders[g][mode])

        for i in range(len(y_t)):
            m = metas[i]
            yt, yp = int(y_t[i]), int(y_p[i])
            rows.append({
                "group": g,
                "split": mode,
                "station_id": int(m.get("station_id", -1)),
                "day_i": int(m.get("day_i", -1)),          # 0-based
                "station_pos": int(m.get("station_pos", -1)),
                "y_true": yt, "y_pred": yp,
                "true_label": LABELS[yt],
                "pred_label": LABELS[yp],
                "correct": (yt == yp),
            })

    df = pd.DataFrame(rows).sort_values(
        ["correct","group","station_id","day_i"],
        ascending=[True, True, True, True]
    ).reset_index(drop=True)

    out_all = CM_DIR / f"predictions_{mode}_station_day_v224.csv"
    out_bad = CM_DIR / f"misclassified_{mode}_station_day_v224.csv"
    df.to_csv(out_all, index=False, encoding="utf-8-sig")
    df[df["correct"] == False].to_csv(out_bad, index=False, encoding="utf-8-sig")

    print("ALL :", out_all)
    print("BAD :", out_bad)
    print(f"Total={len(df)} Bad={(~df['correct']).sum()} Acc={df['correct'].mean():.2%}")
    return df

df_test = export_predictions_station_day("test")
df_test[df_test["correct"] == False].head(20)

ALL : confusion_matrices224/predictions_test_station_day_v224.csv
BAD : confusion_matrices224/misclassified_test_station_day_v224.csv
Total=104004 Bad=41902 Acc=59.71%


,group,split,station_id,day_i,station_pos,y_true,y_pred,true_label,pred_label,correct
0,g1,test,400035,3,11,3,2,Thu,Wed,False
1,g1,test,400035,7,11,3,2,Thu,Wed,False
2,g1,test,400035,11,11,4,2,Fri,Wed,False
3,g1,test,400035,13,11,1,2,Tue,Wed,False
4,g1,test,400035,16,11,1,3,Tue,Thu,False
5,g1,test,400035,17,11,4,5,Fri,Sat,False
6,g1,test,400035,19,11,5,4,Sat,Fri,False
7,g1,test,400035,20,11,3,2,Thu,Wed,False
8,g1,test,400035,21,11,1,5,Tue,Sat,False
9,g1,test,400035,26,11,4,2,Fri,Wed,False
